In [1]:
import mne
import numpy as np
from pathlib import Path

# ---------------- PATHS ----------------
BASE = Path(r"C:\Users\KIIT\Desktop\Parkinson_EEG")
RAW = BASE / "data" / "raw"
OUT = BASE / "results1" / "SanDiego"
OUT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT)

# ------------- SETTINGS ---------------
SF = 256          # assumed sampling rate
SEG_LEN = 2       # seconds per segment
LOW, HIGH = 1, 40 # minimal clinically-used band
USE_NOTCH = True  # set False if not needed

# ------------- STORAGE ----------------
X_segments = []
y_labels = []
subject_ids = []

files = sorted(RAW.rglob("*.bdf"))
print(f"\nTotal EEG files found: {len(files)}\n")

for f in files:
    print("Processing ->", f.name)

    # Load raw EEG
    raw = mne.io.read_raw_bdf(f, preload=True, verbose=False)

    # -------- Minimal Filtering Only --------
    raw.filter(LOW, HIGH, fir_design="firwin", verbose=False)

    if USE_NOTCH:
        raw.notch_filter(freqs=[50, 60], verbose=False)

    data = raw.get_data()      # shape: (channels , samples)
    n_samples = data.shape[1]

    step = SEG_LEN * SF
    n_segments = n_samples // step

    for i in range(n_segments):
        seg = data[:, i*step:(i+1)*step]

        # Store segment
        X_segments.append(seg)

        # Label
        label = 1 if "pd" in f.name.lower() else 0
        y_labels.append(label)

        subject_ids.append(f.stem)

# -------- Convert to Arrays -------
X_segments = np.array(X_segments, dtype=float)
y_labels = np.array(y_labels, dtype=int)
subject_ids = np.array(subject_ids)

print("\nFinal segmented shape:", X_segments.shape)

# --------- SAVE OUTPUT -----------
np.savez(
    OUT / "caseC_segments_minimal.npz",
    X=X_segments,
    y=y_labels,
    subjects=subject_ids,
    fs=SF,
    seg_len=SEG_LEN
)

print("\nSaved ->", OUT / "caseC_segments_minimal.npz")
print("✓ Case-C preprocessing completed.")

Output folder: C:\Users\KIIT\Desktop\Parkinson_EEG\results1\SanDiego

Total EEG files found: 46

Processing -> sub-hc1_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc10_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc18_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc2_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc20_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc21_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc24_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc25_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc29_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc30_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc31_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc32_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc33_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc4_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc7_ses-hc_task-rest_eeg.bdf
Processing -> sub-hc8_ses-hc_task-rest_eeg.bdf
Processing -> sub-pd11_ses-off_task-rest_eeg.bdf
Processing -> sub-pd11_ses-on_task-rest_eeg.bdf
Processing -> sub-pd12_ses-off_task-rest_ee